In [3]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
from src.utils import file_io
from src.config import settings
import re
import unicodedata
import pandas as pd
import numpy as np
pd.options.display.float_format = '{:,.2f}'.format

In [5]:
def limpar_nome_pdf(nome: str) -> str:
    """
    Remove uma ou mais ocorrências de '.pdf' no final do nome (case-insensitive).
    """
    return re.sub(r"(\.pdf)+$", "", nome, flags=re.IGNORECASE)


def listar_pdfs_limpos(root: Path) -> set[str]:
    return {
        limpar_nome_pdf(p.name).lower()
        for p in root.rglob("*.pdf")
    }

def norm_nome(x: str) -> str:
    s = str(x)

    # normalização unicode e espaços estranhos
    s = unicodedata.normalize("NFC", s)
    s = s.replace("\u00A0", " ")
    s = s.strip()

    # se vier caminho completo, pega só o nome
    s = Path(s).name

    # remove qualquer combinação terminal de .pdf e/ou .md
    s = re.sub(r"(\.(pdf|md))+$", "", s, flags=re.IGNORECASE)

    # normaliza espaços
    s = re.sub(r"\s+", " ", s)

    return s.lower()

def listar_mds(root: Path) -> set[str]:
    return {
        norm_nome(p.name)
        for p in root.rglob("*.md")
    }

In [6]:
def listar_pdfs_por_ente(root: Path) -> pd.DataFrame:
    """
    Percorre uma pasta raiz onde cada subpasta representa um ente federativo
    e retorna um DataFrame com:
    - ente_federativo
    - nome_arquivo_pdf
    """

    registros = []

    for pasta_ente in root.iterdir():
        if not pasta_ente.is_dir():
            continue

        ente_federativo = pasta_ente.name

        for pdf in pasta_ente.glob("*.pdf"):
            registros.append({
                "ente_federativo": ente_federativo,
                "nome_arquivo_pdf": pdf.name
            })

    return pd.DataFrame(registros)


def moeda_br_para_float(serie: pd.Series) -> pd.Series:
    def converter(valor):
        if pd.isna(valor):
            return np.nan

        # Se já for número, retorna direto
        if isinstance(valor, (int, float)):
            return float(valor)

        valor_str = str(valor).strip()

        if valor_str == "":
            return np.nan

        # Remove símbolos (R$, espaços etc.)
        valor_str = re.sub(r"[^\d,\.]", "", valor_str)

        # Caso típico BR: tem vírgula decimal
        if "," in valor_str:
            valor_str = valor_str.replace(".", "")
            valor_str = valor_str.replace(",", ".")
            return float(valor_str)

        # Caso sem vírgula:
        # assume que já está no padrão correto
        return float(valor_str)

    return serie.apply(converter)

def padronizar_percentual(serie: pd.Series) -> pd.Series:
    def converter(valor):
        if pd.isna(valor):
            return np.nan

        valor_str = str(valor).strip()

        # Caso com %
        if "%" in valor_str:
            try:
                return float(valor_str.replace("%", "").replace(",", ".")) / 100
            except ValueError:
                return np.nan

        # Caso numérico
        try:
            num = float(valor_str.replace(",", "."))
        except ValueError:
            return np.nan

        # Se for maior que 1, interpretamos como percentual inteiro
        if num > 1:
            return num / 100

        # Caso contrário, já é fração
        return num

    return serie.apply(converter).astype(float)

In [7]:
CAPITAL_FILE_NAME = Path(settings.FINAL_DATA_PATH / settings.CAPITAL_FILE_NAME)
ESTADO_FILE_NAME = Path(settings.FINAL_DATA_PATH/settings.ESTADO_V2_FILE_NAME)
BASE_RAW_MD = Path(settings.INTERIM_DATA_PATH / 'MD')
BASE_RAW_PDF_ESTADO = settings.ESTADO_PDF_ROOT
BASE_RAW_PDF_CAPITAL = settings.CAPITAL_PDF_ROOT

In [8]:
df_capital_old = file_io.load_to_dataframe(CAPITAL_FILE_NAME, sheet_name = 'dados_consolidados_nao_mudar')
df_estado      = file_io.load_to_dataframe(ESTADO_FILE_NAME)

In [9]:
df_estado.columns

Index(['ESTADO', 'path_pdf', 'pdf', 'document', 'chunks_relevantes',
       'texto_completo', 'resultado_llm_dict', 'valor_total', 'cotas_negras',
       'cotas_indigenas', 'cotas_pcd', 'vagas_totais', 'AVALIAÇÃO DO MODELO',
       'OBSERVAÇÕES', 'novo'],
      dtype='object')

In [10]:
COLUNAS_INTERESSE_ESTADO = ['ESTADO', 'pdf', 'cotas_negras', 'cotas_indigenas', 'cotas_pcd', 'vagas_totais','valor_total','novo']
RENAME_COLUNAS_ESTADO = {
    'ESTADO': 'ente_federativo',
    'pdf':'nome_pdf_pk', # chave primária
    'cotas_negras': 'perc_cotas_negras',
    'cotas_indigenas': 'perc_cotas_indigenas',
    'cotas_pcd': 'perc_cotas_pcd',
    'novo': 'is_novo'
}

In [11]:
# RESHAPE
df_estado = df_estado[COLUNAS_INTERESSE_ESTADO]
# SANITIZE
df_estado = df_estado.rename(columns=RENAME_COLUNAS_ESTADO)
# SANITIZE
df_estado[['perc_cotas_negras','perc_cotas_indigenas','perc_cotas_pcd']] = (
    df_estado[['perc_cotas_negras','perc_cotas_indigenas','perc_cotas_pcd']]
    .apply(padronizar_percentual)
)
# SANITIZE
df_estado['valor_total_rs'] = moeda_br_para_float(df_estado['valor_total'])
# SANITIZE  
df_estado['vagas_totais'] = df_estado['vagas_totais'].astype('Int64')

In [237]:
df_estado[df_estado['nome_pdf_pk'] == '2025-05_AMAZONAS_CULTURAVIVA.pdf']

,ente_federativo,nome_pdf_pk,perc_cotas_negras,perc_cotas_indigenas,perc_cotas_pcd,vagas_totais,valor_total,is_novo,valor_total_rs
345,AMAZONAS,2025-05_AMAZONAS_CULTURAVIVA.pdf,0.00,0.50,0.00,2,384988.4,sim,"384,988.40"


In [12]:
df_estado['vagas_totais'].sum()

np.int64(24659)

In [242]:
df_estado['vagas_totais'].describe()

count     338.00
mean       72.96
std       149.40
min         1.00
25%        20.00
50%        36.00
75%        73.75
max     2,114.00
Name: vagas_totais, dtype: Float64

In [245]:
df_estado[df_estado['vagas_totais'] == 2114]

,ente_federativo,nome_pdf_pk,perc_cotas_negras,perc_cotas_indigenas,perc_cotas_pcd,vagas_totais,valor_total,is_novo,valor_total_rs
196,MINAS GERAIS,MG_PRÊMIO_022024.pdf,0.25,0.10,0.05,2114,"R$39.787.500,00",NaN,"39,787,500.00"
